In [1]:
import getpass
import os
from langchain_core.output_parsers import StrOutputParser

#0e87z0rSY7cyjZ8vQ9LLXFyRMYfq8eFj

if not os.environ.get("MISTRAL_API_KEY"):
  os.environ["MISTRAL_API_KEY"] = getpass.getpass("Enter API key for Mistral AI: ")

from langchain.chat_models import init_chat_model

model = init_chat_model("mistral-large-latest", model_provider="mistralai")
output_parser = StrOutputParser()


/Users/charlotte_lac/Documents/LearnVoc/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [4]:
#llm = init_chat_model("gpt-3.5-turbo", model_provider="openai")
#llm = init_chat_model("gpt-4o-mini", model_provider="openai")


output_parser = StrOutputParser()

In [2]:
from langchain_core.prompts import ChatPromptTemplate

#system_template = "Give 3 synonyms for this word: {word}, formatted as a comma-separated list (e.g., fast, quick, speedy)."

#prompt_template = ChatPromptTemplate.from_messages(
#    [("system", system_template), ("user", "{text}")]
#)

In [82]:
system_template = "Give 3 synonyms for this word: {word}, formatted as a comma-separated list such as (fast, quick, speedy)."

prompt_template = ChatPromptTemplate.from_messages(
    [("system", system_template)]
)

word = 'foreshadow'
prompt = prompt_template.invoke({"word": word})

prompt.to_messages()

[SystemMessage(content='Give 3 synonyms for this word: foreshadow, formatted as a comma-separated list such as (fast, quick, speedy).', additional_kwargs={}, response_metadata={})]

In [83]:
response = model.invoke(prompt)
#[resp for resp in response.content]
#response.content.lower().strip("'").split(' ')

In [84]:
response.content #[:3]

'(prefigure, portend, presage)'

In [85]:
list_words = [resp.lower().strip(' ').strip('(').strip(')') for resp in response.content.split(",")]
list_words

['prefigure', 'portend', 'presage']

### Distractors generation

In [86]:
system_template = "Give 2 words that have a different meaning than this list of words: {list_w}, formatted as a comma-separated list such as (fast, quick)."

prompt_template = ChatPromptTemplate.from_messages(
    [("system", system_template)]
)

list_w = list_words + list(word)
prompt = prompt_template.invoke({"list_w": list_w})

#prompt.to_messages()

resp_distractors = model.invoke(prompt)

In [87]:
resp_distractors.content

'Sure, here are two words that have a different meaning than the words in the list: (joyful, serene).'

In [63]:
import re 

def extract_words_in_brackets(text):
    # Use regular expression to find the content within brackets
    match = re.search(r'\(([^)]+)\)', text)
    if match:
        # Split the content by comma and strip any extra whitespace
        words = [word.strip() for word in match.group(1).split(',')]
        return words
    else:
        return None

In [74]:
extract_words_in_brackets(resp_distractors.content)

['joy', 'light']

In [23]:
distractors = [resp.lower().strip(' ') for resp in resp_distractors.content.split(",")]
distractors

['obscure', 'hide']

In [20]:
user_guess = ['obscure', 'hide'] 
[opt for opt in user_guess if opt in list_words]

[]

In [ ]:
text = 'Sure here \n pain, apple'
re.sub(pattern = '')

Testing NLPAUG but it is not efficient

In [9]:
from nlpaug.util.file.download import DownloadUtil
DownloadUtil.download_word2vec(dest_dir='.') # Download word2vec model

Downloading...
From (original): https://drive.google.com/uc?export=download&id=0B7XkCwpI5KDYNlNUTTlSS21pQmM
From (redirected): https://drive.google.com/uc?export=download&id=0B7XkCwpI5KDYNlNUTTlSS21pQmM&confirm=t&uuid=84837faa-9cf1-4f14-8bba-81174d9560dc
To: /Users/charlotte_lac/Documents/LearnVoc/GoogleNews-vectors-negative300.bin.gz
100%|██████████| 1.65G/1.65G [00:50<00:00, 32.7MB/s]


In [ ]:
import nlpaug.augmenter.word as naw
from nlpaug.util import Action
import os
# pip install --use-pep517 gensim==3.8.0

In [29]:
os.environ["MODEL_DIR"] = '../model'
model_dir = '/Users/charlotte_lac/Documents/LearnVoc/'

text = 'The quick brown fox jumps over the lazy dog .'
text = response.content
# Insert word randomly by word embeddings similarity
# model_type: word2vec, glove or fasttext

aug = naw.WordEmbsAug(
    model_type='word2vec', model_path=model_dir+'GoogleNews-vectors-negative300.bin',
    action="insert")
augmented_text = aug.augment(text)

print("Original:")
print(text)
print("Augmented Text:")
print(augmented_text)

Original:
Spotlight, emphasize, showcase
Augmented Text:
['Spotlight, Sandbakken emphasize, caddy showcase']
